In [ ]:
import requests
import json
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from delta import *

EUROSTAT_BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"

def fetch_eurostat(dataset_code: str, filters: dict = {}) -> dict:
    params = {"format": "JSON", "lang": "EN", **filters}
    response = requests.get(f"{EUROSTAT_BASE}/{dataset_code}", params=params)
    response.raise_for_status()
    return response.json()

# Example: fetch GDP data
raw = fetch_eurostat("nama_10_gdp", filters={"geo": "DE,FR,DK,SE,PL", "na_item": "B1GQ"})

# Land in bronze as raw JSON string, partitioned by ingestion date
spark.createDataFrame(
    [(json.dumps(raw),)], ["raw_json"]
).write.format("delta").partitionBy("ingestion_date").save("/bronze/eurostat/gdp")